# 4 · Goal-driven search (BAX)

Sometimes the goal isn't "find the maximum" but "find the **region** of design
space where the outputs satisfy a target." Bayesian Algorithm Execution (BAX) runs
a *goal algorithm* on sampled surrogate functions and focuses sampling to resolve
that target region.

abbot's `BAXPolicy` ports the InfoBAX / SwitchBAX / MeanBAX / US strategies onto
botorch; `Goal` objects (`PercentileSet`, `BandIntersection`, `Wishlist`) define
the target. Here we look for designs whose two outputs land inside a target box.

In [1]:
import torch, warnings
warnings.filterwarnings("ignore")
from abbot import (
    ProblemSpec, ContinuousDomain, OutputSpec, Box,
    BAXPolicy, goals_from_spec, ModelListConstructor, Campaign,
)
torch.manual_seed(0)

In BAX the outputs `a` and `b` are **modeled features a Goal consumes**, not things
to maximize — so we declare them `OBSERVE` (modeled, not optimized). The *target
region* on `(a, b)` is what encodes the intent.

In [2]:
spec = ProblemSpec(
    domain=[ContinuousDomain(name="x", lower=0.0, upper=1.0)],
    outputs=[
        OutputSpec(name="a", direction="OBSERVE"),
        OutputSpec(name="b", direction="OBSERVE"),
    ],
    boxes=[Box(lower=[0.4, 0.4], upper=[0.6, 0.6])],  # target region on (a, b)
)

class TargetExperiment:
    # a(x), b(x) both pass through the [0.4,0.6] box near x=0.5
    def run(self, candidate):
        x = candidate.reshape(-1)
        return torch.cat([x, x])

goal = goals_from_spec(spec)              # Wishlist built from the target box
policy = BAXPolicy(goal, strategy="MeanBAX", n_candidates=200)

### Run the goal-driven campaign
BAXPolicy proposes points from a candidate grid, concentrating where the model believes the target region is.

In [3]:
campaign = Campaign(spec, policy, ModelListConstructor(bounds=spec.bounds_tensor()))
campaign.seed(policy._ensure_grid(spec)[torch.randperm(200)[:6]])
for _ in range(10):
    campaign.step(TargetExperiment(), lambda obs: obs)

In [4]:
# How many sampled designs landed in the target region vs. the base rate?
def in_box(x):
    a, b = float(x), float(x)
    return 0.4 <= a <= 0.6 and 0.4 <= b <= 0.6

shots = [r.candidate for r in campaign.history.records[6:]]  # exclude seeds
hit = sum(in_box(x) for x in shots) / len(shots)
print(f"fraction of BAX shots in the target region: {hit:.2f}")
print("(the target box is ~20% of the x-range, so > 0.2 means BAX concentrated there)")

fraction of BAX shots in the target region: 1.00
(the target box is ~20% of the x-range, so > 0.2 means BAX concentrated there)


### Beyond boxes: arbitrary goals
`Wishlist`/`BandIntersection` cover target boxes, but a goal is just a function
from predicted outputs to target indices. `FunctionGoal` wraps **any**
`fn(f, X) -> indices`, so goals that aren't boxes — level sets, custom top-k,
multi-output logic — plug into the same `BAXPolicy`:

```python
from abbot import FunctionGoal, LevelSet, TopK

# ready-made non-box goals:
goal = LevelSet(threshold=0.5, output=0)      # super-level set on output 0
goal = TopK(k=5, output=0)                    # the 5 best candidates by output 0

# or anything you can express as (values, candidates) -> indices:
goal = FunctionGoal(lambda f, X: (f[:, 0] > f[:, 1]).nonzero(as_tuple=True)[0].tolist())
```

**What this showed:** given a *target region* (or any goal-as-code) rather than a
maximum, BAX steers sampling toward designs that satisfy it. This is the
goal-as-algorithm capability abbot ports from multibax — the same engine that
drives sbido's BAX designers — with `FunctionGoal` giving the arbitrary-goal
flexibility of nn-bax's `FunctionAlgorithm`, now on botorch.